# ERA LTE × climate-toolkit — Colab walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CGIAR-Climate-Data-Hub/climate-toolkit/blob/main/examples/era_lte_colab.ipynb)

This notebook compares the toolkit's climate metrics against ERA long-term experiment (LTE) field records and explores the relationship between yield and climate through visual plots. Using the updated `ERA 2026.1` dataset (`lte_final.csv`) and the `climate_toolkit`, it automates site-specific, multi-theme climate adaptation analyses for agricultural research.

_Notebook authored by **Michel Rwema** — ERA LTE × climate-toolkit integration._


## 1. Set up

In [ ]:
import os
try:
    os.getcwd()
except OSError:
    os.chdir(os.path.expanduser("~"))

%pip install -q climate-toolkit

## 2. Check the toolkit's core modules

In [ ]:
import climate_toolkit as ct

print(f"climate_toolkit v{ct.__version__}")
print("Public API:", [n for n in ct.__all__ if not n.startswith("__")])

## 3. Set up earth engine

In [ ]:
RUN_EARTH_ENGINE = False  # set True once you have registered (free for noncommercial use)
GCP_PROJECT_ID = "your-ee-project-id"  # <-- your registered Cloud project id

# Tip: instead of pasting the id here, store it once in Colab's Secrets
# (key icon in the left sidebar, name it GCP_PROJECT_ID) and use:
#   from google.colab import userdata
#   GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")

if RUN_EARTH_ENGINE:
    import os

    import ee

    ee.Authenticate()  # interactive Google sign-in (works natively in Colab)
    os.environ["GCP_PROJECT_ID"] = GCP_PROJECT_ID
    ee.Initialize(project=GCP_PROJECT_ID)
    print("Earth Engine ready — all sources available.")
else:
    print("Earth Engine disabled — the walkthrough still runs on NASA POWER; switch the section-3 example's --source to nasa_power.")

## 4. Confirm data fetch over your location

In [ ]:
from datetime import date

from climate_toolkit.fetch_data.source_data.sources.utils.models import ClimateVariable

LAT, LON = -1.286, 36.817  # Nairobi, Kenya — swap in your own site

VARS = [
    ClimateVariable.precipitation,
    ClimateVariable.max_temperature,
    ClimateVariable.min_temperature,
]

df = ct.fetch_climate_data(
    source="chirps_v2",
    location_coord=(LAT, LON),
    variables=VARS,
    date_from=date(1991, 1, 1),
    date_to=date(2020, 12, 31),
    verbose=False,
)
df.head()

In [ ]:
# It's a normal DataFrame — plot, resample, export as usual.
df.set_index("date")["precipitation"].plot(
    figsize=(10, 3), title="Daily precipitation, Nairobi 2020 (Nasa Power)"
)

ERA LTE datasets:

1. Dual Table Fetching: Correctly downloads and loads both lte_final.csv (for paired control-treatment comparisons) and lte_arms.csv (for single-treatment time series)
2. Release 2026.1 Column Verification: Added an explicit check to confirm that essential Release 2026.1 columns (LTE.ID, obs_id, yield_t_ha, etc.) are present after loading
.

In [ ]:
# ==============================================================================
# Step 1: Environment Setup & UTF-8 Encoding Fix
# ==============================================================================
import os
import sys
import subprocess
import pandas as pd
from IPython.display import display, Image

# Force UTF-8 output encoding to avoid console crashes on special characters
os.environ["PYTHONUTF8"] = "1"

# Clone repository in Colab or adjust directory if running locally
if os.path.exists("examples/era_lte_workflow.py"):
    pass
elif os.path.exists("era_lte_workflow.py"):
    # Opened from examples/ -> step up to repo root
    %cd ..
else:
    # Fresh Colab environment: clone repo and enter directory
    !git clone -q https://github.com/CGIAR-Climate-Data-Hub/climate-toolkit.git
    %cd climate-toolkit

def run(*args):
    """Run a repo script with the kernel's Python environment quietly."""
    r = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    if r.returncode != 0:
        print("STDOUT:\n", r.stdout)
        print("STDERR:\n", r.stderr)
        raise SystemExit(f"Script failed: {' '.join(args)}")

# ==============================================================================
# Step 2: Fetch and Download Datasets (Release 2026.1)
# ==============================================================================
# 1. Fetch default lte_final.csv via toolkit script
print("Fetching lte_final.csv...")
run('examples/era_fetch_data.py')

# 2. Download lte_arms.csv directly from the ERAgriculture release repository
LTE_ARMS_URL = "https://raw.githubusercontent.com/ERAgriculture/LTEs/main/data/lte_arms.csv"
print(f"Downloading lte_arms.csv from {LTE_ARMS_URL}...")

try:
    run('examples/era_fetch_data.py', '--url', LTE_ARMS_URL, '--out', 'lte_arms.csv')
except Exception:
    print("Fallback: Downloading lte_arms.csv directly with pandas...")
    lte_arms_df = pd.read_csv(LTE_ARMS_URL, low_memory=False)
    lte_arms_df.to_csv("lte_arms.csv", index=False)

print("Data fetch complete.\n")

# ==============================================================================
# Step 3: Load DataFrames into Pandas
# ==============================================================================
# Pairwise comparisons table
lte_final = pd.read_csv("lte_final.csv", low_memory=False)

# Per-treatment arm time series table
lte_arms = pd.read_csv("lte_arms.csv", low_memory=False)

# Corrected: Access shape[0] for rows and shape[1] for columns
print(f"Loaded lte_final.csv: {lte_final.shape[0]:,} rows, {lte_final.shape[1]:,} columns")
print(f"Loaded lte_arms.csv:  {lte_arms.shape[0]:,} rows, {lte_arms.shape[1]:,} columns\n")

# ==============================================================================
# Step 4: Validate Release 2026.1 Column Schema
# ==============================================================================
key_arms_cols = ['LTE.ID', 'obs_id', 'yield_t_ha', 'yield_basis', 'n_means']
key_final_cols = ['LTE.ID', 'obs_id', 'ctrl_id', 'yield_t_ha', 'yield_c_t_ha']

print("Checking Release 2026.1 columns in lte_arms.csv:")
for col in key_arms_cols:
    status = "✓ Present" if col in lte_arms.columns else "✗ Missing"
    print(f"  - {col}: {status}")

print("\nChecking Release 2026.1 columns in lte_final.csv:")
for col in key_final_cols:
    status = "✓ Present" if col in lte_final.columns else "✗ Missing"
    print(f"  - {col}: {status}")

# Preview per-treatment data (including Variety tracking)
print("\nPreview of lte_arms.csv (Single-treatment records):")
display(lte_arms[['LTE.ID', 'Site.ID', 'Product.Simple', 'Variety', 'yield_t_ha', 'yield_basis']].dropna(subset=['Variety']).head())

In [ ]:
# ==============================================================================
# Step 5: Extract Sites with Crop Yield Data using lte_arms.csv
# ==============================================================================
import pandas as pd

# Use lte_arms to capture all unique sites with crop yield data (treatments + controls)
yield_mask = lte_arms["Out.SubInd"] == "Crop Yield"
yield_sites = sorted(
    lte_arms.loc[yield_mask, "Site.ID"].dropna().astype(str).unique()
)

print(
    f"{len(yield_sites)} sites available for yield trends in lte_arms.csv — "
    "pass any (or a distinctive part) to --site:"
)
for s in yield_sites:
    print(f"   {s}")

## Dynamic Yield & Rainfall stress Analysis at the selected site:

Main steps:

1. Set the site and climate source
2. Extract Climate Windows
3. Execute climate_toolkit for rainfall related hazards

In [ ]:
import io
import re
import subprocess
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ==============================================================================
#  CONFIGURATION: SITE QUERY & CLIMATE SOURCE
# ==============================================================================
SITE_QUERY = "Gourton Farm"  # Try "Gourton Farm", "Kouve", "Misamfu ARS", "Askov"
CLIMATE_SOURCE = "era_5"  # Try "era_5" or "nasa_power"
SEASON_DAYS = 150  # Estimated climate window duration
# ==============================================================================


def format_to_mm_dd(val):
  """Converts any date string (YYYY-MM-DD or MM-DD) into 'MM-DD'."""
  if pd.isna(val) or str(val).strip() in ['', 'N/A', 'nan', 'None']:
    return 'N/A'
  s = str(val).strip()
  try:
    dt = pd.to_datetime(s)
    return dt.strftime('%m-%d')
  except Exception:
    if len(s) == 5 and '-' in s:
      return s
    elif len(s) >= 10 and '-' in s:
      parts = s.split('-')
      return f'{parts}-{parts}'
    return s


def calculate_estimated_end(start_mmdd, days=150):
  """Calculates estimated end date (MM-DD) given a start date and duration in days."""
  if start_mmdd == 'N/A':
    return 'N/A'
  try:
    base_dt = pd.to_datetime(f'2000-{start_mmdd}')
    est_dt = base_dt + pd.Timedelta(days=days)
    return est_dt.strftime('%m-%d')
  except Exception:
    return 'N/A'


def parse_hazard_precipitation(cli_output_str):
  """Parses Precipitation value (mm) and Hazard Status from climate_toolkit hazard CLI output."""
  precip_mm = None
  raw_status = 'N/A'
  clean_category = 'Unknown'

  for line in cli_output_str.splitlines():
    # Target the data line containing both "Precipitation" and "mm"
    if 'Precipitation' in line and 'mm' in line:
      val_match = re.search(r'(\d+\.?\d*)\s*mm', line)
      if val_match:
        precip_mm = float(val_match.group(1))

      line_upper = line.upper()
      if 'NO STRESS' in line_upper:
        clean_category = 'No Stress'
      elif 'MODERATE' in line_upper:
        clean_category = 'Moderate Stress'
      elif (
          'SEVERE' in line_upper
          or 'CRITICAL' in line_upper
          or 'HIGH' in line_upper
      ):
        clean_category = 'Severe Stress'
      else:
        clean_category = 'Moderate Stress'

      raw_status = line.strip()
      break

  return precip_mm, clean_category, raw_status


def plot_rainfall_hazard_classification(
    hazard_df, site_name, crop_name, window_str
):
  """Plots seasonal rainfall amounts colored by hazard stress categories (mean line removed)."""
  if hazard_df.empty or 'Precipitation_mm' not in hazard_df.columns:
    print('⚠️ No precipitation hazard data available to plot.')
    return

  df = hazard_df.dropna(subset=['Precipitation_mm']).copy()
  if df.empty:
    print('⚠️ All precipitation values are missing.')
    return

  category_colors = {
      'No Stress': 'forestgreen',
      'Moderate Stress': 'darkorange',
      'Severe Stress': 'crimson',
      'Unknown': 'gray',
  }

  plt.figure(figsize=(13, 6))
  sns.set_style('whitegrid')

  status_handles = {}

  for idx, row in df.iterrows():
    yr = int(row['Year'])
    val = row['Precipitation_mm']
    cat = row['Hazard_Category']
    color = category_colors.get(cat, 'gray')

    plt.bar(
        yr,
        val,
        width=0.45,
        color=color,
        alpha=0.85,
        edgecolor='black',
        linewidth=0.8,
    )
    if cat not in status_handles:
      status_handles[cat] = mpatches.Patch(
          facecolor=color, edgecolor='black', label=cat, alpha=0.85
      )

  plt.title(
      f'{site_name} ({crop_name}): Annual Seasonal Rainfall Hazard'
      f' Assessment\n(Estimated Climate Window: {window_str})',
      fontsize=13,
      fontweight='bold',
      pad=15,
  )
  plt.xlabel('Season Year', fontsize=11, labelpad=8)
  plt.ylabel('Seasonal Total Precipitation (mm)', fontsize=11, labelpad=8)
  plt.xticks(df['Year'].astype(int), rotation=45)

  # Custom legend for Hazard Categories only (mean line removed)
  legend_patches = [
      status_handles[c] for c in category_colors.keys() if c in status_handles
  ]
  plt.legend(
      handles=legend_patches,
      title='Precipitation Hazard Status',
      bbox_to_anchor=(1.02, 1),
      loc='upper left',
      frameon=True,
  )

  plt.tight_layout()
  plot_file = f"{site_name.lower().replace(' ', '_')}_{crop_name.lower().replace(' ', '_')}_rainfall_hazard.png"
  plt.savefig(plot_file, dpi=300, bbox_inches='tight')
  plt.show()
  print(f"✓ Saved Rainfall Hazard plot: '{plot_file}'")


def run_seasonal_rainfall_hazard_analysis(
    site_name, lte_arms_df, source=CLIMATE_SOURCE, season_days=SEASON_DAYS
):
  """Main workflow: Extracts site/crop metadata and calculates annual seasonal precipitation

  hazard classification using climate_toolkit.calculate_hazards.hazards.
  """
  site_mask = (
      lte_arms_df['Site.ID'].str.contains(site_name, case=False, na=False)
  ) & (lte_arms_df['Out.SubInd'] == 'Crop Yield')

  site_data = lte_arms_df[site_mask].copy()

  if site_data.empty:
    print(f"❌ No crop yield records found for site query '{site_name}'.")
    return

  matched_site = str(site_data['Site.ID'].iloc[0])
  crop_col = next(
      (
          c
          for c in site_data.columns
          if c.lower() in ['product.simple', 'product_simple', 'crop']
      ),
      None,
  )
  valid_crops = (
      site_data[crop_col].dropna().unique() if crop_col else [None]
  )

  print('==================================================')
  print(f" Seasonal Rainfall Hazard Workflow: '{matched_site}'")
  print('==================================================\n')

  for crop_val in valid_crops:
    if crop_val is not None and crop_col:
      crop_sub = site_data[site_data[crop_col] == crop_val].copy()
      crop_name = str(crop_val).strip()
    else:
      crop_sub = site_data.copy()
      crop_name = 'Maize'

    if crop_sub.empty:
      continue

    lat_col = next(
        (c for c in crop_sub.columns if c.lower() in ['latitude', 'lat']), None
    )
    lon_col = next(
        (
            c
            for c in crop_sub.columns
            if c.lower() in ['longitude', 'lon', 'long']
        ),
        None,
    )
    lat, lon = None, None
    if lat_col and lon_col:
      v_lats = crop_sub[lat_col].dropna()
      v_lons = crop_sub[lon_col].dropna()
      if not v_lats.empty and not v_lons.empty:
        lat = float(v_lats.iloc[0])
        lon = float(v_lons.iloc[0])

    cw_start_col = next(
        (c for c in crop_sub.columns if c.lower() == 'climate_window_start'),
        None,
    )
    raw_cw_start = (
        crop_sub[cw_start_col].dropna().iloc[0]
        if (cw_start_col and not crop_sub[cw_start_col].dropna().empty)
        else 'N/A'
    )

    rep_cw_start = format_to_mm_dd(raw_cw_start)
    est_cw_end = calculate_estimated_end(rep_cw_start, days=season_days)

    if 'M.Year' in crop_sub.columns:
      crop_sub['Year'] = pd.to_numeric(crop_sub['M.Year'], errors='coerce')
    elif 'm_year_first' in crop_sub.columns:
      crop_sub['Year'] = pd.to_numeric(
          crop_sub['m_year_first'], errors='coerce'
      )

    crop_sub = crop_sub.dropna(subset=['Year']).copy()
    crop_sub['Year'] = crop_sub['Year'].astype(int)

    start_year = int(crop_sub['Year'].min())
    end_year = int(crop_sub['Year'].max())

    print(
        '--------------------------------------------------------------------------'
    )
    print(f' CROP: {crop_name.upper()} ({matched_site})')
    print(
        '--------------------------------------------------------------------------'
    )
    print(f'✓ Coordinates: Latitude = {lat:.4f}, Longitude = {lon:.4f}')
    print(
        f'✓ Estimated Climate Window ({season_days} days): {rep_cw_start} to'
        f' {est_cw_end}'
    )
    print(f'✓ Experiment Period: {start_year} to {end_year}')
    print(f'✓ Climate Source: {source}\n')

    if lat is not None and lon is not None and rep_cw_start != 'N/A':
      toolkit_crop_arg = crop_name.lower().strip()
      fixed_season_arg = f'{rep_cw_start}:{est_cw_end}'

      hazard_records = []
      print(
          ' Calculating annual rainfall hazard via climate_toolkit for each'
          ' season year...'
      )

      for yr in range(start_year, end_year + 1):
        date_from_arg = f'{yr}-01-01'
        date_to_arg = f'{yr}-12-31'

        cli_cmd = (
            f'python -m climate_toolkit.calculate_hazards.hazards'
            f' {toolkit_crop_arg} --location="{lat:.5f},{lon:.5f}"'
            f' --date-from {date_from_arg} --date-to {date_to_arg}'
            f' --fixed-season "{fixed_season_arg}" --source {source} 2>/dev/null'
        )

        output_str = subprocess.getoutput(cli_cmd)
        precip_val, hazard_cat, raw_stat = parse_hazard_precipitation(
            output_str
        )

        hazard_records.append({
            'Year': yr,
            'Precipitation_mm': precip_val,
            'Hazard_Category': hazard_cat,
            'Raw_Status': raw_stat,
        })

      hazard_df = pd.DataFrame(hazard_records)

      print(
          f'\n=== {matched_site} ({crop_name}): Annual Seasonal Rainfall Hazard'
          ' Table ==='
      )
      display(
          hazard_df[['Year', 'Precipitation_mm', 'Hazard_Category']].round(2)
      )

      plot_rainfall_hazard_classification(
          hazard_df, matched_site, crop_name, f'{rep_cw_start} to {est_cw_end}'
      )

    else:
      print('⚠️ Coordinates or climate window unavailable.')


# ==============================================================================
# EXECUTE RAINFALL HAZARD ANALYSIS
# ==============================================================================
run_seasonal_rainfall_hazard_analysis(SITE_QUERY, lte_arms)

---

#  Multi-Theme Climate Adaptation Analysis for Long-Term Experiments (LTE)

##  1. What This Script Does
This notebook automates a **site-specific, multi-theme climate adaptation analysis** on Long-Term Agricultural Experiments (LTE) using the updated **ERA 2026.1 dataset** (`lte_final.csv`) and the **`climate_toolkit`**.

Instead of treating experimental practices in isolation or relying on paper-specific treatment names, this pipeline dynamically detects all experimental **Themes** present at a given trial site (e.g., *Nutrient Management-Soil Management*, *Nutrient Management*, *Soil Management*) and evaluates how standardized farming practices (**`PrName`** / **`SubPrName`**) buffer crop yields against seasonal rainfall hazards over multi-year timelines.

---

##  2. How It Works (Step-by-Step Methodology)

1. **Filtering & Standardization**:
   * Restricts analysis strictly to `Out.SubInd == "Crop Yield"` to avoid mixing crop yields with soil or biomass metrics.
   * Excludes multi-year aggregated rows (`m_year_is_aggregate == 0`) so every annual yield is accurately mapped to its specific 150-day climate window.
2. **Dynamic Theme Discovery**:
   * Scans the dataset for the selected site (e.g., `Gourton Farm`, `Misamfu ARS`, `Kouve`, `Nyabeda`) and identifies the total number of unique themes present.
3. **Identical Management Aggregation**:
   * To prevent confounding bias (as recommended in meta-analysis guidelines), the script averages treatment yields (`MeanT`) and matched control baseline yields (`MeanC`) across identical sub-treatment levels (e.g., averaging No-Till across \\(0\text{N}\\), \\(100\text{N}\\), and \\(200\text{N}\\) against Conventional Tillage across the exact same nitrogen levels).
4. **Climate Toolkit Integration**:
   * Queries ERA5 precipitation data via `climate_toolkit.calculate_hazards.hazards` for the location's coordinates and crop-specific 150-day window, classifying each season into:
     * 🟢 **No Stress**: Normal rainfall conditions.
     * 🟠 **Moderate Stress**: Moderate rainfall deficit / drought hazard.
     * 🔴 **Severe Stress**: Severe or critical rainfall deficit hazard.

---

##  3. What Outputs to Expect

For **each theme** identified at the site, the script generates four sequential outputs:

1. **Verification Table 1 (Annual Reported Averages)**:
   * Displays annual average Treatment yield (`MeanT_avg`), Control yield (`MeanC_avg`), and Yield Advantage (\\(\Delta Y_{\text{avg}} = \text{MeanT} - \text{MeanC}\\)) alongside the underlying paper descriptions (`T.Descrip` & `C.Descrip`) for transparent auditing.
2. **Verification Table 2 (Yields Mapped with Climate Hazards)**:
   * Links the annual yield averages directly with seasonal precipitation totals (\\(\text{mm}\\)) and climate hazard categories.
3. **Summary Table 3 (Normal vs. Stress Comparison)**:
   * Synthesizes overall performance across all years into:
     * **Normal Seasons**: \\(\text{MeanT}_{\text{Normal}}\\), \\(\text{MeanC}_{\text{Normal}}\\), and \\(\Delta Y_{\text{Normal}}\\) (\\(\text{t/ha}\\)).
     * **Stress Seasons**: \\(\text{MeanT}_{\text{Stress}}\\), \\(\text{MeanC}_{\text{Stress}}\\), and \\(\Delta Y_{\text{Stress}}\\) (\\(\text{t/ha}\\)).
4. **Dual-Axis Trend Visualization Plot**:
   * Displays seasonal precipitation as color-coded hazard bars (green/orange/red) on the left axis, overlaid with line trends of practice yield advantages (\\(\Delta Y\\)) on the right axis.

---

##  4. How to Interpret the Results

* **Yield Advantage (\\(\Delta Y = \text{MeanT} - \text{MeanC}\\))**:
  * \\(\Delta Y > 0\\) (\\(\text{t/ha}\\)): The experimental practice produced **higher yields** than conventional control in that season.
  * \\(\Delta Y < 0\\) (\\(\text{t/ha}\\)): The practice suffered a **yield penalty** compared to control.
* **Comparing Normal vs. Stress Performance (\\(\Delta Y_{\text{Normal}}\\) vs. \\(\Delta Y_{\text{Stress}}\\))**:
  * **Climate Buffering Effect**: If \\(\Delta Y_{\text{Stress}} > \Delta Y_{\text{Normal}}\\), the practice provides an even greater relative advantage during dry/drought years than during normal years, demonstrating true climate resilience.
  * **Evaluating Trade-offs**: If a practice has a minor yield penalty during normal years (\\(\Delta Y_{\text{Normal}} < 0\\)) but outperforms control during drought years (\\(\Delta Y_{\text{Stress}} > 0\\)), it represents an adaptation trade-off that risk-averse farmers in dry regions may choose to adopt.

---

###  Colab Usage Tip
To run this analysis on a different LTE site, simply change the `SITE_QUERY` parameter at the top of the code cell (e.g., `SITE_QUERY = "Misamfu ARS"` or `SITE_QUERY = "Kouve"`).

---



In [ ]:
import io
import re
import subprocess
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ==============================================================================
# CONFIGURATION: TARGET SITE & CLIMATE SOURCE
# ==============================================================================
CSV_FILE_PATH = "lte_final.csv"
SITE_QUERY = "Gourton Farm"  # Works for any site: "AREX Training Centre farm, Domboshava", "Kouve", etc.
CLIMATE_SOURCE = "era_5"
SEASON_DAYS = 150
# ==============================================================================


def format_to_mm_dd(val):
  """Converts date string into 'MM-DD'."""
  if pd.isna(val) or str(val).strip() in ["", "N/A", "nan", "None"]:
    return "N/A"
  s = str(val).strip()
  try:
    dt = pd.to_datetime(s)
    return dt.strftime("%m-%d")
  except Exception:
    if len(s) == 5 and "-" in s:
      return s
    elif len(s) >= 10 and "-" in s:
      parts = s.split("-")
      return f"{parts}-{parts}"
    return s


def calculate_estimated_end(start_mmdd, days=150):
  """Calculates estimated end date (MM-DD) given start date and duration."""
  if start_mmdd == "N/A":
    return "N/A"
  try:
    base_dt = pd.to_datetime(f"2000-{start_mmdd}")
    est_dt = base_dt + pd.Timedelta(days=days)
    return est_dt.strftime("%m-%d")
  except Exception:
    return "N/A"


def parse_hazard_precipitation(cli_output_str):
  """Parses Precipitation (mm) and Hazard Status from climate_toolkit output."""
  precip_mm = None
  clean_category = "Unknown"
  for line in cli_output_str.splitlines():
    if "Precipitation" in line and "mm" in line:
      val_match = re.search(r"(\d+\.?\d*)\s*mm", line)
      if val_match:
        precip_mm = float(val_match.group(1))
      line_upper = line.upper()
      if "NO STRESS" in line_upper:
        clean_category = "No Stress"
      elif "MODERATE" in line_upper:
        clean_category = "Moderate Stress"
      elif (
          "SEVERE" in line_upper
          or "CRITICAL" in line_upper
          or "HIGH" in line_upper
      ):
        clean_category = "Severe Stress"
      else:
        clean_category = "Moderate Stress"
      break
  return precip_mm, clean_category


def fetch_climate_hazards_cached(
    lat, lon, crop_name, start_year, end_year, cw_start, cw_end, source, cache
):
  """Fetches climate hazards via climate_toolkit, storing queried years in a cache."""
  fixed_season_arg = f"{cw_start}:{cw_end}"
  toolkit_crop = crop_name.lower().strip()
  for yr in range(start_year, end_year + 1):
    if yr not in cache:
      cli_cmd = (
          f"python -m climate_toolkit.calculate_hazards.hazards"
          f' {toolkit_crop} --location="{lat:.5f},{lon:.5f}" --date-from'
          f' {yr}-01-01 --date-to {yr}-12-31 --fixed-season "{fixed_season_arg}"'
          f" --source {source} 2>/dev/null"
      )
      out_str = subprocess.getoutput(cli_cmd)
      precip_val, hazard_cat = parse_hazard_precipitation(out_str)
      cache[yr] = {
          "Year": yr,
          "Precipitation_mm": precip_val,
          "Hazard_Category": hazard_cat,
      }
  records = [cache[yr] for yr in range(start_year, end_year + 1) if yr in cache]
  return pd.DataFrame(records)


def assign_display_category(row, subpr_count_in_theme):
  """Assigns grouping category: uses SubPrName if multiple SubPrNames exist in the Theme.

  If only 1 SubPrName exists (e.g. 'Inputs N'), parses T.Descrip / C.Descrip
  for Tillage / Treatment categories (e.g., NT, RT, CT).
  """
  subpr = str(row.get("SubPrName", "")).strip()
  if subpr_count_in_theme > 1:
    return subpr
  t_desc = str(row.get("T.Descrip", "")).strip()
  c_desc = str(row.get("C.Descrip", "")).strip()
  text = f"{t_desc} {c_desc}".upper()
  if any(
      k in text for k in ["NT", "NO-TILL", "NO TILL", "NO TILLAGE", "ZERO TILL"]
  ):
    return "No-Till (NT)"
  elif any(
      k in text
      for k in ["RT", "REDUCED TILL", "MIN TILL", "MINTILL", "MINIMUM TILL"]
  ):
    return "Reduced Tillage (RT)"
  elif any(
      k in text for k in ["CT", "CONVENTIONAL", "PLOUGH", "PLOW", "MOULDBOARD"]
  ):
    return "Conventional Tillage (CT)"
  elif t_desc and t_desc.lower() != "nan":
    return f"{subpr} ({t_desc})"
  else:
    return subpr


def run_multi_theme_sequential_analysis(
    csv_path=CSV_FILE_PATH, site_query=SITE_QUERY
):
  """Executes sequential theme-by-theme analysis for any given site in lte_final.csv,

  dynamically resolving SubPrName or T.Descrip/C.Descrip categories.
  """
  df = pd.read_csv(csv_path, low_memory=False)

  # Base Filter for Site, Crop Yield, and Non-aggregated Single Years
  mask_base = (
      (df["Site.ID"].str.contains(site_query, case=False, na=False))
      & (df["Out.SubInd"] == "Crop Yield")
      & (
          df["m_year_is_aggregate"].isna()
          | (df["m_year_is_aggregate"] == 0)
          | (df["m_year_is_aggregate"].eq(False))
      )
  )
  site_df = df[mask_base].copy()
  if site_df.empty:
    print(f"❌ No valid yield records found for site '{site_query}'.")
    return

  # FIXED: Added [0] to all scalar .iloc extractions
  matched_site = str(site_df["Site.ID"].dropna().iloc[0])
  lat = float(site_df["Latitude"].dropna().iloc[0])
  lon = float(site_df["Longitude"].dropna().iloc[0])

  cw_start_col = next(
      (c for c in site_df.columns if c.lower() == "climate_window_start"),
      None,
  )
  raw_cw_start = (
      site_df[cw_start_col].dropna().iloc[0]
      if (cw_start_col and not site_df[cw_start_col].dropna().empty)
      else "09-01"
  )
  rep_cw_start = format_to_mm_dd(raw_cw_start)
  est_cw_end = calculate_estimated_end(rep_cw_start, days=SEASON_DAYS)

  crop_name = (
      str(site_df["Product.Simple"].dropna().iloc[0])
      if (
          "Product.Simple" in site_df.columns
          and not site_df["Product.Simple"].dropna().empty
      )
      else "Maize"
  )

  # Clean Numeric Columns
  site_df["Year"] = pd.to_numeric(
      site_df.get("M.Year", site_df.get("m_year_first")), errors="coerce"
  )
  site_df["MeanT"] = pd.to_numeric(site_df["MeanT"], errors="coerce")
  site_df["MeanC"] = pd.to_numeric(site_df["MeanC"], errors="coerce")
  site_df = site_df.dropna(subset=["Year", "MeanT", "MeanC"]).copy()
  site_df["Year"] = site_df["Year"].astype(int)

  # Identify All Unique Themes at this Site
  themes_list = [
      t for t in site_df["Theme"].dropna().unique() if str(t).strip() != ""
  ]
  total_themes = len(themes_list)
  print(
      "=========================================================================================="
  )
  print(f" MULTI-THEME SEQUENTIAL ANALYSIS PIPELINE — SITE: '{matched_site}'")
  print(f"   Total Themes Identified: {total_themes}")
  for idx, t in enumerate(themes_list, 1):
    print(f"   • Theme {idx}: {t}")
  print(
      f"   Crop: {crop_name} | Coordinates: ({lat:.4f}, {lon:.4f}) | Climate"
      f" Window: {rep_cw_start} to {est_cw_end}"
  )
  print(
      "==========================================================================================\n"
  )
  climate_cache = {}

  # Iterate Theme by Theme
  for theme_idx, theme_name in enumerate(themes_list, 1):
    theme_sub = site_df[site_df["Theme"] == theme_name].copy()
    subpr_count = theme_sub["SubPrName"].nunique()

    theme_sub["Display_Category"] = theme_sub.apply(
        lambda r: assign_display_category(r, subpr_count), axis=1
    )

    print(
        "=========================================================================================="
    )
    print(f" THEME {theme_idx} OF {total_themes}: '{theme_name}'")
    print(
        f"   Site: {matched_site} | Records: {len(theme_sub)} |"
        f" Categories: {list(theme_sub['Display_Category'].unique())}"
    )
    print(
        "==========================================================================================\n"
    )

    # Verification Table 1
    yearly_verify_df = (
        theme_sub.groupby(["PrName", "SubPrName", "Display_Category", "Year"])
        .agg(
            MeanT_avg=("MeanT", "mean"),
            MeanC_avg=("MeanC", "mean"),
            N_Obs=("MeanT", "count"),
            T_Descrip_Included=(
                "T.Descrip",
                lambda x: " | ".join(sorted(list(set(x.dropna())))),
            ),
            C_Descrip_Included=(
                "C.Descrip",
                lambda x: " | ".join(sorted(list(set(x.dropna())))),
            ),
        )
        .reset_index()
    )
    yearly_verify_df["ΔY_avg (t/ha)"] = (
        yearly_verify_df["MeanT_avg"] - yearly_verify_df["MeanC_avg"]
    )

    print(
        f" VERIFICATION TABLE 1 (Theme {theme_idx}): Annual Reported Averages"
        " (MeanT & MeanC)"
    )
    cols_verify = [
        "PrName",
        "SubPrName",
        "Display_Category",
        "Year",
        "N_Obs",
        "MeanT_avg",
        "MeanC_avg",
        "ΔY_avg (t/ha)",
        "T_Descrip_Included",
        "C_Descrip_Included",
    ]
    display(
        yearly_verify_df[cols_verify]
        .sort_values(["PrName", "Display_Category", "Year"])
        .round(2)
    )

    # Fetch Climate Hazards
    start_year = int(yearly_verify_df["Year"].min())
    end_year = int(yearly_verify_df["Year"].max())

    hazard_df = fetch_climate_hazards_cached(
        lat,
        lon,
        crop_name,
        start_year,
        end_year,
        rep_cw_start,
        est_cw_end,
        CLIMATE_SOURCE,
        climate_cache,
    )

    # Verification Table 2
    merged_annual = pd.merge(yearly_verify_df, hazard_df, on="Year", how="left")

    print(
        f"\n VERIFICATION TABLE 2 (Theme {theme_idx}): Annual Yields Mapped"
        " with Climate Hazards"
    )
    cols_hazard_display = [
        "PrName",
        "SubPrName",
        "Display_Category",
        "Year",
        "Precipitation_mm",
        "Hazard_Category",
        "MeanT_avg",
        "MeanC_avg",
        "ΔY_avg (t/ha)",
    ]
    display(merged_annual[cols_hazard_display].round(2))

    # Summary Table 3 (Includes Adaptation_Buffering)
    stress_cats = ["Moderate Stress", "Severe Stress"]
    normal_df = merged_annual[merged_annual["Hazard_Category"] == "No Stress"]
    stress_df = merged_annual[
        merged_annual["Hazard_Category"].isin(stress_cats)
    ]

    adaptation_summary = []
    unique_pr_cats = merged_annual[
        ["PrName", "SubPrName", "Display_Category"]
    ].drop_duplicates()

    for _, row_tax in unique_pr_cats.iterrows():
      p_name, sp_name, cat_name = (
          row_tax["PrName"],
          row_tax["SubPrName"],
          row_tax["Display_Category"],
      )

      sub_norm = normal_df[
          (normal_df["PrName"] == p_name)
          & (normal_df["Display_Category"] == cat_name)
      ]
      sub_stress = stress_df[
          (stress_df["PrName"] == p_name)
          & (stress_df["Display_Category"] == cat_name)
      ]

      y_norm_t = sub_norm["MeanT_avg"].mean() if not sub_norm.empty else np.nan
      y_norm_c = sub_norm["MeanC_avg"].mean() if not sub_norm.empty else np.nan
      delta_norm = y_norm_t - y_norm_c if pd.notna(y_norm_t) else np.nan

      y_stress_t = (
          sub_stress["MeanT_avg"].mean() if not sub_stress.empty else np.nan
      )
      y_stress_c = (
          sub_stress["MeanC_avg"].mean() if not sub_stress.empty else np.nan
      )
      delta_stress = y_stress_t - y_stress_c if pd.notna(y_stress_t) else np.nan

      adaptation_buffering = (
          (delta_stress - delta_norm)
          if (pd.notna(delta_stress) and pd.notna(delta_norm))
          else np.nan
      )

      adaptation_summary.append({
          "Theme": theme_name,
          "PrName": p_name,
          "SubPrName": sp_name,
          "Practice Category": cat_name,
          "MeanT Normal (t/ha)": (
              round(y_norm_t, 2) if pd.notna(y_norm_t) else None
          ),
          "MeanC Normal (t/ha)": (
              round(y_norm_c, 2) if pd.notna(y_norm_c) else None
          ),
          "ΔY Normal (t/ha)": (
              round(delta_norm, 2) if pd.notna(delta_norm) else None
          ),
          "MeanT Stress (t/ha)": (
              round(y_stress_t, 2) if pd.notna(y_stress_t) else None
          ),
          "MeanC Stress (t/ha)": (
              round(y_stress_c, 2) if pd.notna(y_stress_c) else None
          ),
          "ΔY Stress (t/ha)": (
              round(delta_stress, 2) if pd.notna(delta_stress) else None
          ),
          "Adaptation_Buffering (t/ha)": (
              round(adaptation_buffering, 2)
              if pd.notna(adaptation_buffering)
              else None
          ),
      })

    summary_df = pd.DataFrame(adaptation_summary)

    print(
        f"\n SUMMARY TABLE 3 (Theme {theme_idx}): Yield & Delta Yield Summary"
        " (Normal vs Stress)"
    )
    display(summary_df)

    # Plotting Trends
    fig, ax1 = plt.subplots(figsize=(14, 6))
    sns.set_style("white")

    category_colors = {
        "No Stress": "forestgreen",
        "Moderate Stress": "darkorange",
        "Severe Stress": "crimson",
    }

    years = hazard_df["Year"].values
    precip = hazard_df["Precipitation_mm"].values
    cats = hazard_df["Hazard_Category"].values

    for idx, yr in enumerate(years):
      ax1.bar(
          yr,
          precip[idx],
          width=0.4,
          color=category_colors.get(cats[idx], "gray"),
          alpha=0.45,
          edgecolor="black",
      )

    ax1.set_xlabel("Season Year", fontsize=11, labelpad=8)
    ax1.set_ylabel(
        "Seasonal Precipitation (mm)",
        fontsize=11,
        fontweight="bold",
        color="black",
    )
    ax1.set_ylim(0, max(precip) * 1.15)
    ax1.set_xticks(years)

    ax2 = ax1.twinx()
    unique_cats = merged_annual["Display_Category"].unique()
    palette = sns.color_palette("tab10", n_colors=len(unique_cats))

    for idx, cat_name in enumerate(unique_cats):
      sub_data = merged_annual[merged_annual["Display_Category"] == cat_name]
      if not sub_data.empty:
        ax2.plot(
            sub_data["Year"],
            sub_data["ΔY_avg (t/ha)"],
            label=f"ΔY: {cat_name}",
            color=palette[idx % len(palette)],
            linewidth=2.5,
            marker="o",
        )

    ax2.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.2,
        label="Control Baseline (ΔY=0)",
    )
    ax2.set_ylabel(
        "Yield Advantage ΔY (t/ha)",
        fontsize=11,
        fontweight="bold",
        color="darkblue",
    )
    ax2.grid(False)

    plt.title(
        f"{matched_site} ({crop_name}) — Theme {theme_idx}: {theme_name}\nYield"
        " Advantage (ΔY = MeanT_avg - MeanC_avg) vs Climate Hazards",
        fontsize=12,
        fontweight="bold",
        pad=15,
    )

    hazard_patches = [
        mpatches.Patch(
            facecolor=c, alpha=0.45, label=lab, edgecolor="black", linewidth=0.8
        )
        for lab, c in category_colors.items()
        if lab in cats
    ]
    ax1.legend(
        handles=hazard_patches,
        title="Rainfall Hazard",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
    )
    ax2.legend(
        title="Practice Categories ΔY",
        loc="upper left",
        bbox_to_anchor=(1.02, 0.55),
    )

    plt.tight_layout()
    slug_site = matched_site.lower().replace(" ", "_")
    plot_filename = f"{slug_site}_theme_{theme_idx}_adaptation.png"
    plt.savefig(plot_filename, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved plot for Theme {theme_idx}: '{plot_filename}'\n\n")


# Execute Multi-Theme Pipeline
run_multi_theme_sequential_analysis(CSV_FILE_PATH, SITE_QUERY)